### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [21]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = True  # True for single worker, False for multiple workers

### Start with our Message class

In [22]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [23]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [24]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [25]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [26]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [27]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [28]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [29]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some reasons in favor of choosing AutoGen for your new AI Agent project:

1. **Improved Efficiency**: AutoGen employs asynchronous messaging and an event-driven architecture, which allows for faster and more efficient communication between agents. This can lead to quicker response times and overall better performance in task execution.

2. **Scalability**: The framework is designed with modularity and extensibility in mind, enabling the creation of scalable and customizable systems. This is beneficial for projects that may need to grow or adapt to changing requirements over time.

3. **Enhanced Collaboration**: AutoGen supports better collaboration between agents, facilitating complex interactions and workflows that can benefit from multiple agents working together seamlessly.

4. **Reduced Development Time**: With built-in tools and features, AutoGen can help streamline the development process, reducing the time it takes to build and deploy AI agents.

5. **Flexibility**: The framework allows for various configurations and customizations, providing the flexibility to tailor the agents to specific project needs.

These benefits make AutoGen a strong candidate for developing AI Agent systems effectively and efficiently. 

TERMINATE

## Cons of AutoGen:
Here are some cons of using AutoGen in an AI agent project:

1. **Steep Learning Curve**: Users often find the learning curve challenging and may feel overwhelmed trying to master AutoGen's capabilities.

2. **User Difficulty**: Some users struggle with the interface and tools, which can lead to frustration and hinder productivity.

3. **Lack of Intuitiveness**: AutoGen may not be very intuitive, making it difficult for new users to navigate and effectively utilize its features.

4. **Complexity**: The complexity of the system may deter teams that prefer simpler or more straightforward solutions for their AI needs.

5. **Limited Documentation**: In some cases, the available documentation may not be sufficient, leading to further challenges during implementation and development.

These factors could potentially impact the usability and effectiveness of AutoGen in your project. 

TERMINATE



## Decision:

After considering the pros and cons of AutoGen, I recommend moving forward with its implementation for your AI agent project. 

The strong advantages, including improved efficiency, scalability, and enhanced collaboration, outweigh the challenges associated with the steep learning curve and complexity. The benefits of faster communication and reduced development time are critical for the project's success, especially if the project anticipates growth and adaptability in the future. 

While the usability issues and documentation limitations are valid concerns, investing in training and support for the team can mitigate these drawbacks. Overall, AutoGen appears to align well with the project's goals and requirements.

TERMINATE

In [30]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [31]:
await host.stop()